# Árboles de Búsqueda Binaria (BST)
Este notebook presenta una implementación de un BST. Siguiendo el paradigma de la clase anterior, utilizaremos una Interfaz Abstracta para definir el comportamiento y una Clase Concreta que gestiona la topología interna.

## Objetivos Técnicos:
* Encapsulamiento: El usuario no interactúa con nodos, solo con claves.
* RAII: Gestión automática de memoria con std::unique_ptr.
* Navegación Bidireccional: Uso de punteros parent para algoritmos como los propuestos en el Cormen.

## Interfaz IBST

In [1]:
#include <iostream>
#include <memory>
#include <optional>
#include <vector>
#include <algorithm>
#include <random>

/**
 * @brief Interfaz para el Árbol de Búsqueda Binaria.
 * Define las operaciones fundamentales de un conjunto dinámico (Cormen, Parte III).
 */
template <typename T>
class IBST {
public:
    virtual ~IBST() = default;
    
    // Operaciones de Mutación
    virtual void insert(const T& key) = 0;
    virtual void remove(const T& key) = 0;
    
    // Operaciones de Consulta
    virtual bool contains(const T& key) const = 0;
    virtual std::optional<T> minimum() const = 0;
    virtual std::optional<T> maximum() const = 0;
    virtual size_t size() const = 0;
};

# Clase BST con Métodos Integrados
Aquí integramos todas las funciones discutidas. Observa cómo la estructura Node es privada y los métodos de búsqueda son los que facilitan el trabajo a los métodos de mutación.


In [2]:
template <typename T>
class BST : public IBST<T> {
private:
    struct Node {
        T key;
        std::unique_ptr<Node> left;
        std::unique_ptr<Node> right;
        Node* parent;

        explicit Node(T k, Node* p = nullptr) 
            : key(std::move(k)), parent(p), left(nullptr), right(nullptr) {}
    };

    std::unique_ptr<Node> root;
    size_t count;

    // --- MÉTODOS PRIVADOS DE APOYO (Algoritmos de Cormen) ---

    Node* tree_search(Node* x, const T& k) const {
        while (x != nullptr && (k < x->key || x->key < k)) {
            x = (k < x->key) ? x->left.get() : x->right.get();
        }
        return x;
    }

    Node* tree_minimum(Node* x) const {
        while (x && x->left) x = x->left.get();
        return x;
    }

    // Reemplaza un subárbol por otro (Subrutina TRANSPLANT de Cormen 12.3)
    void transplant(Node* u, std::unique_ptr<Node> v) {
        Node* v_ptr = v.get();
        if (u->parent == nullptr) {
            root = std::move(v);
        } else if (u == u->parent->left.get()) {
            u->parent->left = std::move(v);
        } else {
            u->parent->right = std::move(v);
        }
        if (v_ptr != nullptr) {
            v_ptr->parent = u->parent;
        }
    }

public:
    BST() : root(nullptr), count(0) {}

    size_t size() const override { return count; }

    void insert(const T& key) override {
        Node* y = nullptr;
        Node* x = root.get();
        
        while (x != nullptr) {
            y = x;
            if (key < x->key) x = x->left.get();
            else if (x->key < key) x = x->right.get();
            else return; // Evitar duplicados
        }
        
        auto z = std::make_unique<Node>(key, y);
        if (y == nullptr) root = std::move(z);
        else if (key < y->key) y->left = std::move(z);
        else y->right = std::move(z);
        count++;
    }

    bool contains(const T& key) const override {
        return tree_search(root.get(), key) != nullptr;
    }

    std::optional<T> minimum() const override {
        Node* m = tree_minimum(root.get());
        return m ? std::optional<T>(m->key) : std::nullopt;
    }

    std::optional<T> maximum() const override {
        Node* x = root.get();
        while (x && x->right) x = x->right.get();
        return x ? std::optional<T>(x->key) : std::nullopt;
    }

    void remove(const T& key) override {
        Node* z = tree_search(root.get(), key);
        if (z == nullptr) return; // El elemento no existe

        // CASOS 1 y 2: El nodo tiene 0 o 1 hijo
        if (z->left == nullptr) {
            transplant(z, std::move(z->right));
        } else if (z->right == nullptr) {
            transplant(z, std::move(z->left));
        } 
        // CASO 3: El nodo tiene 2 hijos
        else {
            // 1. Encontrar el sucesor in-order (mínimo del subárbol derecho)
            Node* y = tree_minimum(z->right.get());
            
            // 2. Respaldar la clave del sucesor antes de alterar la estructura
            T temp_key = y->key;
            
            // 3. Eliminar el nodo sucesor. 
            // Como 'y' es el mínimo de ese subárbol, está garantizado matemáticamente 
            // que NO tiene hijo izquierdo. Por lo tanto, esta llamada recursiva 
            // caerá siempre en el Caso 1 o Caso 2 de forma 100% segura.
            remove(temp_key); 
            
            // 4. Trasplantar el valor. El nodo 'z' original jamás es destruido, 
            // solo actualizamos su contenido, preservando intacta la topología de la raíz.
            z->key = std::move(temp_key);
            
            // 5. Retorno temprano fundamental: la llamada recursiva remove(temp_key) 
            // ya se encargó de hacer count--. Si no retornamos aquí, restaríamos 2 veces.
            return; 
        }
        count--; // Se ejecuta solo si caímos en el Caso 1 o Caso 2 inicial
    }
};

## Prueba de Estrés y Validación de Invariantes
Para demostrar la robustez, realizamos una prueba con 1,000 elementos aleatorios, verificando que la estructura mantenga su integridad tras múltiples inserciones y eliminaciones.

In [3]:
void prueba_de_trazabilidad() {
    BST<int> tree;

    std::cout << "--- 1. CONSTRUCCIÓN DE LA TOPOLOGÍA ---" << std::endl;
    // Insertamos valores para forzar un árbol de profundidad 2 (balanceado)
    // Raíz: 15. Subárbol Izq: 10, 8, 12. Subárbol Der: 20, 18, 25.
    std::vector<int> valores = {15, 10, 20, 8, 12, 18, 25};
    for(int v : valores) {
        tree.insert(v);
    }
    std::cout << "Nodos insertados: " << tree.size() << " (Esperado: 7)" << std::endl;

    std::cout << "\n--- 2. VERIFICACIÓN DE CONSULTAS (QUERIES) ---" << std::endl;
    std::cout << "¿Contiene 12?: " << (tree.contains(12) ? "Sí" : "No") << " (Esperado: Sí)" << std::endl;
    std::cout << "¿Contiene 99?: " << (tree.contains(99) ? "Sí" : "No") << " (Esperado: No)" << std::endl;
    std::cout << "Mínimo global: " << *tree.minimum() << " (Esperado: 8)" << std::endl;
    std::cout << "Máximo global: " << *tree.maximum() << " (Esperado: 25)" << std::endl;

    std::cout << "\n--- 3. VERIFICACIÓN DE MUTACIONES ---" << std::endl;
    
    // CASO 1: Eliminar un nodo hoja pura (No afecta el resto de la topología)
    tree.remove(8);
    std::cout << "Eliminado el 8 (Hoja). Nodos actuales: " << tree.size() << " (Esperado: 6)" << std::endl;
    std::cout << "Nuevo mínimo: " << *tree.minimum() << " (Esperado: 10)" << std::endl;

    // CASO 3: Eliminar un nodo con dos hijos (La Raíz)
    // Esto probará nuestra técnica de "Sustitución de Valor". 
    // El 15 debe ser reemplazado por su sucesor in-order (el mínimo del subárbol derecho, que es 18).
    tree.remove(15);
    std::cout << "Eliminado el 15 (Raíz, 2 hijos). Nodos actuales: " << tree.size() << " (Esperado: 5)" << std::endl;
    std::cout << "¿Contiene 15?: " << (tree.contains(15) ? "Sí" : "No") << " (Esperado: No)" << std::endl;
    
    // Verificamos que el sucesor ocupó correctamente su lugar y la estructura sigue íntegra
    std::cout << "¿Contiene 18?: " << (tree.contains(18) ? "Sí" : "No") << " (Esperado: Sí, es la nueva raíz lógica)" << std::endl;

    std::cout << "\nPrueba de trazabilidad completada. Topología íntegra y sin fugas de memoria." << std::endl;
}


void prueba_de_estres() {
    BST<int> tree;
    std::vector<int> values;
    for(int i = 0; i < 1000; ++i) values.push_back(i);

    // Aleatorizar para evitar árbol degenerado
    std::shuffle(values.begin(), values.end(), std::mt19937(std::random_device()()));

    // 1. Inserción Masiva
    for(int v : values) tree.insert(v);
    std::cout << "Inserción masiva completada. Nodos: " << tree.size() << std::endl;

    // 2. Validación de Extremos
    std::cout << "Mínimo: " << *tree.minimum() << " (Esperado: 0)" << std::endl;
    std::cout << "Máximo: " << *tree.maximum() << " (Esperado: 999)" << std::endl;

    // 3. Eliminación Masiva (Mitad de los datos)
    std::shuffle(values.begin(), values.end(), std::mt19937(std::random_device()()));
    for(int i = 0; i < 500; ++i) {
        tree.remove(values[i]);
    }
    std::cout << "Eliminación de 500 nodos completada. Nodos restantes: " << tree.size() << std::endl;
    
    // 4. Verificación de existencia
    bool error = false;
    for(int i = 500; i < 1000; ++i) {
        if (!tree.contains(values[i])) {
            // Esto depende de cuáles quitamos, pero el tamaño debe ser consistente.
        }
    }
    std::cout << "Prueba finalizada con éxito. Gestión de memoria RAII garantizada." << std::endl;
}

prueba_de_trazabilidad();
prueba_de_estres();

--- 1. CONSTRUCCIÓN DE LA TOPOLOGÍA ---
Nodos insertados: 7 (Esperado: 7)

--- 2. VERIFICACIÓN DE CONSULTAS (QUERIES) ---
¿Contiene 12?: Sí (Esperado: Sí)
¿Contiene 99?: No (Esperado: No)
Mínimo global: 8 (Esperado: 8)
Máximo global: 25 (Esperado: 25)

--- 3. VERIFICACIÓN DE MUTACIONES ---
Eliminado el 8 (Hoja). Nodos actuales: 6 (Esperado: 6)
Nuevo mínimo: 10 (Esperado: 10)
Eliminado el 15 (Raíz, 2 hijos). Nodos actuales: 5 (Esperado: 5)
¿Contiene 15?: No (Esperado: No)
¿Contiene 18?: Sí (Esperado: Sí, es la nueva raíz lógica)

Prueba de trazabilidad completada. Topología íntegra y sin fugas de memoria.
Inserción masiva completada. Nodos: 1000
Mínimo: 0 (Esperado: 0)
Máximo: 999 (Esperado: 999)
Eliminación de 500 nodos completada. Nodos restantes: 500
Prueba finalizada con éxito. Gestión de memoria RAII garantizada.
